# sPTC on a real GPU, v3: regression A/B, fairer baselines, saturation, general workload

Measures the speculation-aware scheduler plugin (`sptc_vllm`) against stock vLLM 0.32 on
**Qwen3-1.7B (fp16) on one Kaggle T4**. The simulator's predictions for every
measurement were committed before this notebook (`results/prereg_v3/` and its B4 addendum
`results/prereg_v3b/`, commit messages `prereg: ...`); the last cells score the run against
them, and also against the earlier `results/prereg_v2` (unchanged).

What is new against v2 (`sptc-bundle-v2.zip`): the plugin code has the audit fixes (vLLM
adapter and SGLang adapter), the plugin tests in section A are larger, and section **B4**
adds two fairer baselines: vLLM priority with only the bets at low priority,
and a client-side cap on concurrent bets on stock vLLM (its cap is the plugin's free window,
8, fixed in advance).

**Settings:** Accelerator **GPU T4** (one is enough), Internet **on**.
**Input:** the dataset made from `sptc-bundle-v3.zip` (`kaggle/make_bundle.sh`). Attach only
this bundle (the cell below picks the one that has `results/prereg_v3`). Optional input when
resuming: the `sptc_out_v3.zip` of an earlier session of this notebook, uploaded as a
dataset. Do not resume from `sptc_out_v2.zip`: those runs used the plugin before the audit
fixes.

**Expected runtimes** (estimates: simulator session times x 1.3 plus ~20 s start-up per run
and 4 min per server launch, `vllm_bench.py plan`; upper bounds rather than means):

| section | what | estimate |
|---|---|---|
| setup | install vLLM 0.32, spec-ptc, the plugin | ~8 min |
| A | tests (plugin incl. `tests_vllm`, simulator, bench on the fake server), scheduler overhead | ~30-35 min, CPU mostly |
| B1 | regression A/B, compiled: stock vs plugin fixed / adaptive / off, 1/4/8 agents, 3 repeats (1 agent: 5), each server launched twice (second round in reverse order), plus up to 3 unmeasured cache-filling launches | ~1 h 55 min |
| B2 | the same at 1 agent on `--enforce-eager` launches (allowance modes with the code held equal), each server launched twice | ~50 min |
| B3 | vLLM `--scheduling-policy priority` at 8 agents, with stock FCFS launched next to it, 3 repeats each | ~30 min |
| **A + B1-B3 (core)** | | **~4 h** |
| S | the plugin's smoke test S1-S3 on its own eager launch (S3 rewritten after the audit, #41, never run on a server) | ~10 min |
| B4 | optional, new: bets-only priority and a client-side cap on bets, next to stock FCFS with and without bad guesses and the plugin, 8 and 4 agents, 3 repeats, two launches per server | ~1 h (~45 min at 8 agents only) |
| C1 | saturation sweep, core preset (1-24 agents; reuses B's runs) | ~1 h 10 min |
| C2 | ReAct / background workload, quick matrix | ~25 min |
| C3 | saturation sweep, full preset (adds 2, 12, 32 agents and two arms) | ~2 h |
| C4 | ReAct / background workload, full matrix | ~2 h |

Section C is ordered by value: run as far as the session allows. Every run (sweep) and
every cell (ReAct matrix) is saved as soon as it ends, and a rerun skips what is finished.
`/kaggle/working/sptc_out_v3.zip` (with a copy of the predictions inside) is refreshed at
the end of each section and, in the background, every 10 minutes. The last two cells
score the run against `results/prereg_v3` (with `prereg_v3b`, and `prereg_v2`) and zip
everything.

**Keep the zip when the session ends.** Everything is about 10.5-11 h, close to Kaggle's
12 h session limit, and the estimates are simulator-based, not measured. The zip lives in
`/kaggle/working`, which an interactive session keeps after it ends only if the notebook's
**Persistence** setting is on ("Files only"). So either turn Persistence on before
starting, or stop after
C2 (about 6.5 h: setup, A, B1-B3, S, B4, C1, C2) and download `sptc_out_v3.zip`, then
resume in a new session with the zip attached.

In [ ]:
import os, subprocess, sys, time, json, glob, shutil, zipfile

REPO = ""   # optional GitHub URL; leave empty to use the uploaded dataset

def bundles(top="/kaggle/input"):
    # folders that contain both sptc-sim and spec-ptc, at any depth
    return [root for root, dirs, files in os.walk(top) if "sptc-sim" in dirs and "spec-ptc" in dirs]

def find_bundle(top="/kaggle/input"):
    found = bundles(top)
    if not found:  # zip uploaded but not unpacked: unpack it into /kaggle/working
        for z in glob.glob(f"{top}/**/*.zip", recursive=True):
            if "sptc_out" in os.path.basename(z):
                continue  # an earlier session's output, not a bundle
            with zipfile.ZipFile(z) as f:
                f.extractall("/kaggle/working/_unzipped")
        found = bundles("/kaggle/working/_unzipped")
    v3 = [b for b in found if os.path.isdir(f"{b}/sptc-sim/results/prereg_v3")]
    return v3[0] if v3 else None

if REPO:
    B = "/kaggle/working/sptc-bundle"
    subprocess.run(["git", "clone", "--depth", "1", REPO, f"{B}/sptc-sim"], check=True)
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/alexzhang13/spec-ptc",
                    f"{B}/spec-ptc"], check=True)
else:
    src = find_bundle()
    if src is None:
        print("No v3 bundle (sptc-sim/results/prereg_v3) found. Here is what is under /kaggle/input:")
        for root, dirs, files in os.walk("/kaggle/input"):
            if root.count(os.sep) <= 6:
                print(root, dirs[:8], files[:5])
        raise SystemExit("bundle not found: see the listing above")
    print("found bundle at", src)
    B = "/kaggle/working/sptc-bundle"
    shutil.copytree(src, B, dirs_exist_ok=True)  # inputs are read-only

SIM, SPTC = f"{B}/sptc-sim", f"{B}/spec-ptc"
OUT = "/kaggle/working/sptc_out_v3"
os.makedirs(OUT, exist_ok=True)

# resume: an earlier session's output, if attached (finished runs are then skipped)
for z in glob.glob("/kaggle/input/**/sptc_out_v3*.zip", recursive=True):
    with zipfile.ZipFile(z) as f:
        f.extractall("/kaggle/working/_resume")
for d in glob.glob("/kaggle/input/**/sptc_out_v3", recursive=True) + \
         glob.glob("/kaggle/working/_resume/**/sptc_out_v3", recursive=True):
    print("carrying over", d)
    shutil.copytree(d, OUT, dirs_exist_ok=True)

import threading
_zip_lock = threading.Lock()

def _zip():
    """Copy the predictions next to the results; sptc_out_v3 -> sptc_out_v3.zip through a
    temporary file, so the zip on disk is never half-written."""
    with _zip_lock:
        for v in ("prereg_v3", "prereg_v3b", "prereg_v2"):
            shutil.copytree(f"{SIM}/results/{v}", f"{OUT}/{v}", dirs_exist_ok=True)
        tmp = "/kaggle/working/_sptc_out_v3.tmp.zip"
        if os.path.exists(tmp):
            os.remove(tmp)
        r = subprocess.run(["nice", "-n", "19", "zip", "-qr", tmp, "sptc_out_v3"],
                           cwd="/kaggle/working")
        if r.returncode not in (0, 18):  # 18: a file vanished while zipping (a run being saved)
            raise RuntimeError(f"zip exit code {r.returncode}")
        os.replace(tmp, "/kaggle/working/sptc_out_v3.zip")

def save(note=""):
    """Refresh the zip (called at the end of every section)."""
    _zip()
    print(note, "saved:", os.path.getsize("/kaggle/working/sptc_out_v3.zip") // 1024, "KiB")

def _autosave(every_s=600):
    # during the long cells too: a session that ends mid-cell keeps its finished runs
    while True:
        time.sleep(every_s)
        try:
            _zip()
        except Exception as e:  # never kill the thread
            print("autosave failed:", e)

if not any(t.name == "sptc-autosave" for t in threading.enumerate()):
    threading.Thread(target=_autosave, name="sptc-autosave", daemon=True).start()

save("start")
print(SIM, sorted(os.listdir(SIM)))
print(open(f"{SIM}/results/prereg_v3/predictions.md").read()[:600])
!nvidia-smi

In [ ]:
%%bash -s "$SPTC" "$SIM"
pip -q install "vllm==0.32.0" 2>&1 | tail -2
pip -q install pandas numpy pytest 2>&1 | tail -1
pip -q install -e "$1" 2>&1 | tail -1                          # spec-ptc (the sptc-hints version)
pip -q install -e "$2/integrations" --no-deps 2>&1 | tail -1   # sptc_vllm plugin
# Kaggle preinstalls torchaudio built for an older CUDA; vLLM's torch upgrade makes it fail
# at import (transformers imports it), so remove it: nothing here uses audio
pip -q uninstall -y torchaudio 2>&1 | tail -1
python -c "import torchvision.ops, torchvision; print('torchvision', torchvision.__version__, 'OK')"
python -c "import vllm, torch, spec_ptc; print('vllm', vllm.__version__, 'torch', torch.__version__, torch.cuda.get_device_name(0))"

## A. Tests and scheduler overhead (CPU; no server running)

1. The plugin's tests: its local policy tests and the vLLM-dependent `tests_vllm` (the
   step-by-step differential test against stock vLLM on untagged traffic, the
   `prefill_allowance` adaptive/off cases, the startup-log check, and
   `test_sptc_overhead.py`, which prints the plugin's per-step CPU cost: hence `-s`).
2. The simulator's tests, and the benchmark clients against the fake OpenAI server
   (end to end with spec-ptc, resume, the A/B rounds and warm-up pass).
3. `overhead_bench.py vllm`: `schedule()` + `update_from_output()` per step, stock vs plugin,
   sync and async, 8 to 512 running requests. Scored against declared bounds (section A of
   `predictions.md`).

The local plugin tests now include the vLLM adapter on a stand-in of vLLM 0.32's scheduler
(skipped where vLLM is installed, so here) and the SGLang adapter on fakes; the ones that
need the vendored SGLang sources skip (they are not in the bundle). `tests_vllm` grew with
the audit fixes: the differential test drives vLLM's batch-queue loop and has a
default-like config with prefix caching (157 cases instead of 56). The SGLang adapter's
tests (`integrations/tests_sglang`) need SGLang and are not run here.

In [ ]:
!cd {SIM}/integrations && python -m pytest -q -s -p no:cacheprovider tests tests_vllm 2>&1 | tee {OUT}/tests_plugin.txt | tail -60
!cd {SIM} && SPTC_SPEC_PTC={SPTC} python -m pytest -q -p no:cacheprovider tests kaggle/tests 2>&1 | tee {OUT}/tests_sim_bench.txt | tail -15
save("A tests")

In [ ]:
# scheduler overhead (CPU only; ~5-10 min)
!python {SIM}/kaggle/overhead_bench.py vllm --out {OUT}/overhead 2>&1 | tail -45
!python {SIM}/kaggle/overhead_bench.py pure --out {OUT}/overhead 2>&1 | tail -15
save("A overhead")

## B. Core: the regression A/B and the priority baseline

All runs: the October 19 load test's workload (spec-ptc turns of `map16`, `map_reduce`,
`two_stage`, `side_branch_chain`, 4 per agent, the root stream served by the same vLLM), all
agents released together. `vllm_bench.py sweep --launch` starts and stops each server itself:

- **Stock, priority and plugin-fixed servers carry the same `--additional-config`**, so they
  compile and run one torch.compile artifact (it is part of vLLM's compile hash;
  `--scheduler-cls` is not). The adaptive and off servers compile their own; B2 compares
  the three modes with the code held equal (`--enforce-eager`).
- **No measurement on a fresh compile.** A new session starts with an empty compile cache.
  A launch that compiles fresh is stopped unmeasured and launched again, once per compiled
  artifact, so every measured launch loads a cached one (`role: cache_fill` in
  `sweep/launches.jsonl`; `results/t4_run1_analysis/root_cause.md`, protocol step 1).
- **Each server is launched twice, the second round in reverse order** (B1: stock, fixed,
  adaptive, off, then off, adaptive, fixed, stock; B2 the same on eager launches; B3:
  priority, stock, stock, priority). Repeat r runs on launch r % 2, so stock vs stock
  across launches is the noise floor, and no arm always runs first or last.
- **Same prompts on every arm.** Each run's prompt prefix names the client workload, load
  and repeat, not the arm (no prefix-cache reuse across runs on one launch). Sub-calls
  stop at EOS, so the summary also gives sub-call tokens per session against stock.
- After every measured launch: 3 warm-up calls and one throwaway 1-agent run (`sweep/warm/`).
- Every launch's compile state (AOT artifact saved or loaded, the plugin's startup line)
  and GPU state go to `sweep/launches.jsonl`; GPU clocks, temperature and power are
  sampled during every run (nvidia-smi).
- A server that fails to start is recorded in `launches.jsonl` and skipped; the rest of the
  preset still runs. Rerun the cell to retry it.

In [ ]:
BENCH = f"python {SIM}/kaggle/vllm_bench.py"
COMMON = f"--model Qwen/Qwen3-1.7B --out {OUT} --spec-ptc {SPTC} --launch"
for p in ("ab", "ab_eager", "prio8", "baselines", "core", "full"):
    !{BENCH} plan --preset {p} --out {OUT}/plan 2>&1 | head -1
!python {SIM}/kaggle/bench_general.py estimate --quick --out {OUT}/plan | tail -1
!python {SIM}/kaggle/bench_general.py estimate --out {OUT}/plan | tail -1
!nvidia-smi --query-gpu=memory.used --format=csv   # must be ~0: no other server running

In [ ]:
# B1: regression A/B, compiled (~1 h 55 min). Safe to rerun after a crash: finished runs are skipped.
!{BENCH} sweep --preset ab {COMMON} 2>&1 | grep -v "^INFO" | tail -150
save("B1")

In [ ]:
# B2: the same at 1 agent on --enforce-eager launches, two per server (~50 min)
!{BENCH} sweep --preset ab_eager {COMMON} 2>&1 | grep -v "^INFO" | tail -60
save("B2")

In [ ]:
# B3: vLLM priority scheduling at 8 agents, stock FCFS launched next to it (~30 min)
!{BENCH} sweep --preset prio8 {COMMON} 2>&1 | grep -v "^INFO" | tail -40
!grep -h "sptc scheduler" {OUT}/sweep/server_*.log | sort | uniq -c
!{BENCH} report --out {OUT} 2>&1 | head -80
!python {SIM}/experiments/prereg_compare.py --out {OUT} 2>&1 | head -12
save("B3")

## S. The plugin's smoke test on vLLM 0.32 (~10 min)

`integrations/kaggle/sptc_vllm_smoke.py`, cases S1-S3, on its own launch: the plugin with
`--enforce-eager` and `free_window` 2, Qwen3-1.7B as in the first run's smoke test, port
8100 (the sweeps use 8000). S1: an idle bet completes. S2: a bet is rejected while two
streams decode. S3: an `early` call of ~3000 prompt tokens arrives while a normal stream
decodes, untagged and tagged in alternation (3 each), each pass with its own prompt; the
check is that the prefill allowance shortens the stream's longest stall (median over the
passes). S3 was rewritten after the audit (#41: both passes used to send one prompt, so
the tagged pass hit the prefix cache) and has not run on a server since. Its stall check is
a measurement and can fail on its own; it is not a pre-registered claim.

In [ ]:
# S: smoke S1-S3 on its own eager launch of the plugin (~10 min)
import urllib.request
SMOKE_URL = "http://localhost:8100/v1"
cmd = ["vllm", "serve", "Qwen/Qwen3-1.7B", "--dtype", "half", "--max-model-len", "8192",
       "--gpu-memory-utilization", "0.90", "--port", "8100", "--enforce-eager",
       "--scheduler-cls", "sptc_vllm.scheduler.SptcScheduler",
       "--additional-config", json.dumps({"sptc": {"free_window": 2}})]
env = {**os.environ, "PYTHONPATH": os.pathsep.join(
    x for x in (f"{SIM}/integrations", os.environ.get("PYTHONPATH")) if x)}
with open(f"{OUT}/smoke_vllm_server.log", "w") as log:
    srv = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env)
    try:
        for _ in range(450):  # up to 15 min
            if srv.poll() is not None:
                raise RuntimeError("smoke server exited; see smoke_vllm_server.log")
            try:
                urllib.request.urlopen(SMOKE_URL + "/models", timeout=3)
                break
            except OSError:
                time.sleep(2)
        else:
            raise TimeoutError("smoke server not up after 15 min")
        r = subprocess.run([sys.executable, f"{SIM}/integrations/kaggle/sptc_vllm_smoke.py",
                            "--base-url", SMOKE_URL, "--model", "Qwen/Qwen3-1.7B",
                            "--cases", "s1,s2,s3"], capture_output=True, text=True, timeout=1800)
        with open(f"{OUT}/smoke_vllm_s123.jsonl", "w") as f:
            f.write(r.stdout)
        print(r.stdout[-6000:], r.stderr[-1500:], "exit", r.returncode)
    finally:
        srv.terminate()
        try:
            srv.wait(timeout=120)
        except subprocess.TimeoutExpired:
            srv.kill()
        time.sleep(10)  # let the GPU memory go
!grep -c Traceback {OUT}/smoke_vllm_server.log || true
!grep -h "sptc scheduler" {OUT}/smoke_vllm_server.log | head -2
save("S smoke")

## B4 (optional). Fairer baselines (~1 h; ~45 min at 8 agents only)

Two baselines for the question "is a scheduler change needed at all?". Both run with one
bad guess per speculation (`spec+bad`), 3 repeats, at 8 agents and then 4, each server
launched twice (fcfs, priority, plugin, then in reverse):

- `prio-bets/spec+bad`: vLLM `--scheduling-policy priority` with **only the bets** at
  priority 1. Early calls stay at the default, as the plugin treats them (the B3 arms put
  early calls at priority 1 too, which biases against priority).
- `proxy/spec+bad`: stock FCFS behind a **client-side cap**: a bet is sent only while fewer
  than 8 requests (every class) are in flight to the server, which is the plugin's
  sequence-count rule as a proxy can see it. `proxy-sem/spec+bad`: a plain semaphore of 8
  on bets in flight. A refused bet is never sent and never queued. The cap is the plugin's
  free window, fixed in advance, not tuned. The gate runs inside the benchmark client, so
  it adds no forwarding hop: a best case for a proxy.
- `fcfs-b4/spec+bad` and `plugin-b4/spec+bad`: stock FCFS and the plugin, launched in the
  same rounds. The B4 ratios use `fcfs-b4` (same launches as the proxy arms).
- `fcfs-b4/spec`: stock FCFS **without bad guesses**, on the same launches. Every bet in
  this bench is a worthless decoy, so this is what a gate that refuses every bet gets: the
  best any gate can do here.
- **Same prompts on every B4 arm.** The four arms that share a stock FCFS launch send the
  same prompts as the others and are kept out of each other's prefix cache by vLLM's
  `cache_salt` (if the server refuses it, the sweep prints a warning and numbers their
  prefixes instead). The B4 table in `sweep/summary.md` gives, per arm, sub-call tokens per
  session against `fcfs-b4` (away from 1: not the same work), session p50 against no bets,
  decoys refused, and the prefix-cache hit share.

Simulated (P1, `results/prereg_v3`, claims B4-*, B4P-*, B4R-*; `results/prereg_v3b`, claims
B4C-*, B4N-*): the plugin's session p50 is 0.98x (4 agents) and 0.99x (8 agents) of the
window cap's, and 0.91x / 0.93x of the semaphore's; bets-only priority equals FCFS (1.00x).
The window cap and the plugin are within 3% of no bets (1.05x / 1.03x at 4 agents, 1.00x /
0.99x at 8), because they refuse 92-100% of the bets. **What B4 can and cannot show:**
whether a client-side cap gets the T4 win without a scheduler change on a workload whose
bets are all worthless. It cannot show any cost of refusing too many bets, so it cannot
tell a smart gate from "drop every bet"; at 8 agents the window cap and the plugin both
refuse every bet in the simulation, and B4P-3/4 then compare two arms that send none. The
in-scheduler parts matter only under KV pressure or a waiting queue. Set `RUN_B4 = False`
to skip; `B4_AGENTS = "8"` for 8 agents only. `--max-num-queued-reqs` (a class-blind cap in vLLM's API server) is not run: it
refuses root streams and normal calls too, so the benchmark's sessions would fail.

In [ ]:
# B4 (optional): bets-only priority and a client-side cap on bets (~1 h; 8 agents only ~45 min)
RUN_B4 = True
B4_AGENTS = "8,4"
if RUN_B4:
    !{BENCH} sweep --preset baselines --agents {B4_AGENTS} {COMMON} 2>&1 | grep -v "^INFO" | tail -60
    !{BENCH} report --out {OUT} > /dev/null 2>&1
    !sed -n '/^## Baselines (B4)/,/^## Against stock/p' {OUT}/sweep/summary.md | head -24
    # every B4 arm should show the same prefix; the arms that share a stock launch, their
    # own cache_salt (None: no salt, or the server refused it; the sweep then warns above)
    B4 = {"fcfs-b4/spec+bad", "proxy/spec+bad", "proxy-sem/spec+bad", "prio-bets/spec+bad",
          "plugin-b4/spec+bad", "fcfs-b4/spec"}
    for f in sorted(glob.glob(f"{OUT}/sweep/runs/*__n8__r0.json")):
        x = json.load(open(f))
        if x["arm"] in B4:
            print(x["arm"], repr(x["salt"]), x.get("cache_salt"))
    save("B4")

## C. Extended, ordered by value (run as far as the session allows)

- **C1** saturation sweep, `core` preset: fcfs / prio / plugin x nospec, spec, spec+bad at
  1, 4, 8, 16, 24 agents, 3 repeats. Runs B already made (same arm, load and repeat) are
  skipped. Each arm's ladder stops once it is past saturation.
- **C2** ReAct agents with next-call guesses, and background batch jobs
  (`bench_general.py matrix --quick`): stock FCFS, priority and plugin servers in turn.
  One run per cell and one launch per arm: its verdicts are indicative. A rerun keeps
  finished cells.
- **C3** the `full` preset: adds `prio/nospec`, `plugin/nospec` and 2, 12, 32 agents.
- **C4** the full ReAct / background matrix (all modes, 1-16 agents, 16 batch workers).

In [ ]:
# C1: saturation sweep, core (~1 h 10 min beyond what B ran)
!{BENCH} sweep --preset core {COMMON} 2>&1 | grep -v "^INFO" | tail -150
save("C1")

In [ ]:
# C2: ReAct / background workload, quick matrix (~25 min; port 8000 must be free)
!python {SIM}/kaggle/bench_general.py matrix --model Qwen/Qwen3-1.7B --out {OUT}/general_quick --quick 2>&1 | grep -v "^INFO" | tail -40
save("C2")

In [ ]:
# C3: saturation sweep, full (~2 h more; resumes from core)
!{BENCH} sweep --preset full {COMMON} 2>&1 | grep -v "^INFO" | tail -150
save("C3")

In [ ]:
# C4: ReAct / background workload, full matrix (~2 h)
!python {SIM}/kaggle/bench_general.py matrix --model Qwen/Qwen3-1.7B --out {OUT}/general_full 2>&1 | grep -v "^INFO" | tail -60
save("C4")

## Score against the pre-registered predictions, and download

`experiments/prereg_compare.py` applies the rules fixed in `results/prereg_v3/predictions.md`
(the same rules as v2) to whatever ran: every claim is supported, refuted, unresolved,
invalid (host CPU saturated or a failed run) or not run. It scores `prereg_v3` first and
then its B4 addendum `prereg_v3b` and `prereg_v2`, unchanged, below it. Output: `sptc_out_v3/prereg_compare.md` and `.json`,
next to `sweep/summary.md` (the measured tables). Run these two cells at any point.

The pre-registered rule supports a ratio on its point estimate alone. For "no gap" claims
(AB-1, AB-2, the allowance modes) read the **strict** column too: it needs the ratio's CI
over server launches inside the tolerance. The report also checks the prediction files'
sha256 against their prereg commits (v3: 2e39a4d, v3b: a403a94, v2: 1ecddb5), and adds
per-pattern ratios, priority vs the adjacent stock launches and, per ratio claim, a 'work'
column (sub-call tokens per session, numerator over denominator) as secondary analyses.
Read B4's claims with `results/prereg_v3b/predictions.md` ("How to read B4"): v3's B4 rule
text names fcfs-b4 as every denominator, but B4P-1..8 divide by the proxy arms (scoring
uses each claim's own denominator).

In [ ]:
!{BENCH} report --out {OUT} 2>&1 | tail -3
!python {SIM}/experiments/prereg_compare.py --out {OUT} 2>&1 | head -140

In [ ]:
save("final")
!ls -la /kaggle/working/sptc_out_v3.zip